In [ ]:
from pyspark.sql.functions import lit, col

In [ ]:
dbutils.widgets.text("catalog", "olist_project_dev")

dbutils.widgets.text("bronze_schema", "olist_bronze")
dbutils.widgets.text("raw_olist_geolocation_table", "olist_geolocation")

dbutils.widgets.text("silver_schema", "olist_silver")
dbutils.widgets.text("geolocation_table", "geolocation_silver")

In [ ]:
catalog = dbutils.widgets.get("catalog")

bronze_schema = dbutils.widgets.get("bronze_schema")
raw_olist_geolocation_table_name = dbutils.widgets.get("raw_olist_geolocation_table")

silver_schema = dbutils.widgets.get("silver_schema")
geolocation_table_name = dbutils.widgets.get("geolocation_table")

In [ ]:
raw_olist_geolocation_df = spark.table(f"{catalog}.{bronze_schema}.{raw_olist_geolocation_table_name}")

In [ ]:
if not spark.catalog.tableExists(f"{catalog}.{silver_schema}.{geolocation_table_name}"):
    spark.sql(
        f"""
        CREATE TABLE {catalog}.{silver_schema}.{geolocation_table_name} (
            geolocationZipCodePrefix INT,
            geolocationLatitude DOUBLE,
            geolocationLongitude DOUBLE,
            geolocationCity STRING,
            geolocationState STRING
        )
        TBLPROPERTIES (
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact' = 'true'
        )
        """
    )

In [ ]:
geolocation_silver_df = (
    raw_olist_geolocation_df
    .where(col("geolocation_zip_code_prefix").isNotNull())
    .select(
        col("geolocation_zip_code_prefix").alias("geolocationZipCodePrefix"),
        col("geolocation_latitude").alias("geolocationLatitude"),
        col("geolocation_longitude").alias("geolocationLongitude"),
        col("geolocation_city").alias("geolocationCity"),
        col("geolocation_state").alias("geolocationState")
    )
)

In [ ]:
geolocation_silver_df.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{silver_schema}.{geolocation_table_name}")